# 04 · Videoevents und Trackingkontakte vergleichen

Die gespeicherten T-DEED-Vorhersagen aus Notebook 03 werden mit den ursprünglichen Kontaktkandidaten und bestehenden Besitzgrenzen verglichen. Ergebnisse sind zeitliche Diagnosen und eine Featuretabelle für Notebook 05. Aktionsqualität und Teamkontrolle sind damit nicht unabhängig validiert.

[← Notebook 03](../03_videoaktionen/03_videoaktionen.ipynb) · [Notebook 05 →](../05_fusionspilot/05_fusionspilot.ipynb)

In [ ]:
from pathlib import Path
from collections import defaultdict
import copy
import json
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from IPython.display import display

## 1. Einstellungen

Vier feste Zeitfenster: ±0,2 / ±0,5 / ±1 / ±2 s. Das Hauptfenster beträgt ±0,5 s. Die Negativkontrolle verwendet 1.000 Verschiebungen und Seed 42. Alle Klassen werden gemeinsam eins zu eins zugeordnet: zuerst maximale Paarzahl, dann kleinster Gesamtabstand.

In [ ]:
PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "tactical/BAR-ATH_tactical_atd.txt").is_file())

PILOT = PROJECT_DIR / "output/video_event_spotting_pilot"

OUT = PILOT / "integration_diagnostics"

for part in ("tables", "figures", "manifests"):
    (OUT / part).mkdir(parents=True, exist_ok=True)

FIG = OUT / "figures"

TABLES = OUT / "tables"

RANDOM_STATE, N_SHIFTS, MIN_SHIFT_S = 42, 1000, 2.4

WINDOWS_S, FEATURE_WINDOW_S = [0.2, 0.5, 1.0, 2.0], 0.5

FPS, N_FRAMES = 25, 199319

FFPROBE = shutil.which("ffprobe")

if FFPROBE is None:
    raise RuntimeError("Die vorhandene ffprobe-Installation wird benötigt.")
print("Zeitfenster [s]:", WINDOWS_S, "| Negativkontrollen:", N_SHIFTS, "| Seed:", RANDOM_STATE)

## 2. Vorhersagen und gespeicherte Scores laden

Die summierten Scores werden wie im Original über die unterstützenden Fenster gemittelt. Die ursprüngliche klassenweise Soft-NMS zeigt die Verdichtung von Rasterkandidaten zu finalen Vorhersagen. Schwelle 0,2 und Radius 12 bleiben erhalten.

In [ ]:
clips = pd.read_csv(PILOT / "diagnostics/clip_manifest.csv")

video = pd.read_csv(PILOT / "predictions/video_events_pilot.csv", keep_default_na=False)

video = video.rename(columns={"tracking_frame": "frame", "event_type": "video_event_type", "event_confidence": "video_event_score"})

video = video.sort_values(["source_clip", "frame", "video_event_type"], kind="stable").reset_index(drop=True)

video.insert(0, "event_id", [f"V{i:03d}" for i in range(len(video))])

video = video.merge(clips[["scene_id", "source_clip", "first_frame", "stop_frame_exclusive"]], on="source_clip", validate="many_to_one")

settings = json.loads((PILOT / "manifests/inference_runtime.json").read_text())["settings"]

STRIDE, SCORE_THRESHOLD, SNMS_RADIUS = settings["stride"], settings["threshold"], settings["soft_nms_window_sampled_frames"]

SOURCE = PROJECT_DIR / "artifacts/video_event_spotting/source/d953b4de8a2ad941c565e1683c5b07255cbe644e"

classes = (SOURCE / "data/soccernetball/class.txt.txt").read_text().splitlines()

score_data = {scene.scene_id: dict(np.load(PILOT / f"diagnostics/model_scores/{scene.scene_id}.npz", allow_pickle=False))
              for scene in clips.itertuples()}

def soft_non_maximum_supression(pred, window, threshold=0.01):
    preds = copy.deepcopy(pred)
    new_pred = []
    for video_pred in preds:
        events_by_label = defaultdict(list)
        for e in video_pred['events']:
            events_by_label[e['label']].append(e)
        events = []
        i = 0
        for v in events_by_label.values():
            if type(window) is not list:
                class_window = window
            else:
                class_window = window[i]
                i += 1
            while len(v) > 0:
                e1 = max(v, key=lambda x: x['score'])
                if e1['score'] < threshold:
                    break
                pos1 = [pos for pos, e in enumerate(v) if e['frame'] == e1['frame']][0]
                events.append(copy.deepcopy(e1))
                list_pos = [pos for pos, e in enumerate(v) if (e['frame'] >= e1['frame'] - class_window) & (e['frame'] <= e1['frame'] + class_window)]
                for pos in list_pos:
                    v[pos]['score'] = v[pos]['score'] * np.abs(e1['frame'] - v[pos]['frame']) ** 2 / (class_window + 0) ** 2
                v.pop(pos1)
        events.sort(key=lambda x: x['frame'])
        new_video_pred = copy.deepcopy(video_pred)
        new_video_pred['events'] = events
        new_video_pred['num_events'] = len(events)
        new_pred.append(new_video_pred)
    return new_pred

raster_rows, reconstructed_rows, stage_rows = [], [], []

for scene in clips.itertuples():
    arrays = score_data[scene.scene_id]
    scores = arrays["summed_scores"].copy()
    scores /= np.maximum(arrays["support"], 1)[:, None]
    indices, label_indices = np.where(scores[:, 1:] >= SCORE_THRESHOLD)
    candidates = [{"frame": int(i), "label": classes[j], "score": float(scores[i, j + 1])} for i, j in zip(indices, label_indices)]
    events = soft_non_maximum_supression([{"events": candidates}], window=SNMS_RADIUS, threshold=SCORE_THRESHOLD)[0]["events"]
    for stage, records in [("pre_soft_nms_grid", candidates), ("post_soft_nms", events)]:
        for event in records:
            row = {"scene_id": scene.scene_id, "frame": scene.first_frame + event["frame"] * STRIDE,
                   "video_event_type": event["label"], "video_event_score": event["score"], "stage": stage}
            (raster_rows if stage == "pre_soft_nms_grid" else reconstructed_rows).append(row)
    stage_rows.append({"scene_id": scene.scene_id, "pre_nms_grid_candidates": len(candidates), "post_nms_events": len(events),
                       "grid_positions": len(scores), "window_covered_positions": int((arrays["window_coverage"] > 0).sum()),
                       "score_supported_positions": int((arrays["support"] > 0).sum()),
                       "window_starts": arrays["window_starts"].tolist()})

raw_candidates = pd.DataFrame(raster_rows)

reconstructed = pd.DataFrame(reconstructed_rows)

raw_candidates.to_csv(TABLES / "pre_soft_nms_grid_candidates.csv", index=False)

pd.DataFrame(stage_rows).to_csv(TABLES / "event_processing_stages.csv", index=False)

print("Gespeicherte Videoevents:", len(video))



display(pd.DataFrame(stage_rows))

## 3. Ereignisdichte und Scores

Klassenhäufigkeiten und Abstände werden innerhalb desselben Clips ausgewertet. Benachbarte Vorhersagen sind ohne unabhängige Referenz nicht automatisch Duplikate.

In [ ]:
counts = pd.crosstab(video.scene_id, video.video_event_type).reindex(clips.scene_id, fill_value=0)
confidence = video.groupby("video_event_type").video_event_score.describe()
intervals = video.sort_values(["scene_id", "frame", "video_event_type"], kind="stable").copy()
intervals["previous_any_class_gap_s"] = intervals.groupby("scene_id").frame.diff() / FPS
intervals["previous_same_class_gap_s"] = intervals.groupby(["scene_id", "video_event_type"]).frame.diff() / FPS
intervals[["event_id", "scene_id", "frame", "video_event_type", "previous_any_class_gap_s", "previous_same_class_gap_s"]].to_csv(TABLES / "event_intervals.csv", index=False)
close_pairs = intervals.loc[intervals.previous_same_class_gap_s.le(SNMS_RADIUS * STRIDE / FPS)].copy()
close_pairs[["event_id", "scene_id", "frame", "video_event_type", "previous_same_class_gap_s"]].to_csv(TABLES / "close_same_class_predictions.csv", index=False)
counts.to_csv(TABLES / "event_counts_by_clip_class.csv")
confidence.to_csv(TABLES / "event_score_distributions.csv")
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
counts.plot.bar(stacked=True, ax=axes[0], rot=25)
axes[0].set(title="Vorhersagen je Clip", ylabel="Anzahl", xlabel="Automatischer Kontext")
for name, group in video.groupby("video_event_type"):
    axes[1].hist(group.video_event_score, bins=np.linspace(.2, 1, 13), alpha=.5, label=name)
axes[1].set(title="Unkalibrierte Soft-NMS-Scores", xlabel="Score", ylabel="Vorhersagen")
axes[1].legend()
axes[2].hist(intervals.previous_same_class_gap_s.dropna(), bins=np.arange(0, 12.5, .5))
axes[2].set(title="Abstand gleicher Klasse im selben Clip", xlabel="Zeitabstand [s]", ylabel="Benachbarte Eventpaare")
fig.savefig(FIG / "01_event_distributions.png", dpi=150)
plt.show()
display(counts)
display(confidence.round(3))
print("Beobachtung: Die Rasterkandidaten wurden zu finalen Events verdichtet; die CSV enthält keine identischen Clip/Frame/Klasse-Schlüssel.")

## 4. Ursprüngliche Trackingkontakte rekonstruieren

RTS-Glättung und Kontaktsuche laufen auf vollständigen beobachteten Ballruns, damit Clipgrenzen die Glättung nicht verändern. Parameter stammen aus dem vorhandenen Kontaktmanifest. Der ursprüngliche Zustandsautomat ordnet die Kontakte in die bestehende Besitzfolge ein.

In [ ]:
contact_04 = pd.read_csv(PROJECT_DIR / "artifacts/upstream/contact_classification/submissions/contact_state_shared_D.csv")

CONTACT_DIR = PROJECT_DIR / 'artifacts/upstream/contact_classification'
config = json.loads((CONTACT_DIR / 'run_manifest.json').read_text())
PITCH = np.asarray(config['pitch_m'], dtype=float)
NEAR_DISTANCE_M = config['near_distance_m']
PEAK_ACCELERATION_M_S2 = config['peak_acceleration_m_s2']
MIN_RUN_SECONDS = config['min_run_seconds']
EDGE_GUARD_SECONDS = config['edge_guard_seconds']
MEASUREMENT_SIGMA_M = config['measurement_sigma_m']
PROCESS_SIGMA_M_S2 = config['process_sigma_m_s2']
META_PATH = PROJECT_DIR / 'tactical/BAR-ATH_tactical_atd.xml'
RAW_PATH = PROJECT_DIR / 'tactical/BAR-ATH_tactical_atd.txt'
root = ET.parse(META_PATH).getroot()
team_nodes = root.findall('./Metadata/Teams/Team')
team_map = {team.get('id'): 'H' if i == 0 else 'A' for i, team in enumerate(team_nodes)}
channel_to_track = {channel.get('id'): channel.get('playerId') for channel in root.findall('./Metadata/PlayerChannels/PlayerChannel')}
spec = root.find('./DataFormatSpecifications/DataFormatSpecification')
track_ids = [channel_to_track[group.find('PlayerChannelRef').get('playerChannelId')] for group in spec.findall('./SplitRegister/SplitRegister')]
player_team = {player.get('id'): team_map[player.get('teamId')] for player in root.findall('./Metadata/Players/Player')}
track_sides = np.asarray([player_team[track_id] for track_id in track_ids])
positions = np.empty((N_FRAMES, len(track_ids) + 1, 2), dtype=float)
with RAW_PATH.open() as handle:
    for frame, line in enumerate(handle):
        raw_frame, players_raw, ball_raw = line.strip().split(':')
        values = np.fromstring(players_raw.replace(';', ',') + ',' + ball_raw, sep=',').reshape(-1, 3)
        positions[frame] = values[:, :2] * PITCH
player_xy_contact = positions[:, :-1]
ball_xy_contact = positions[:, -1]
ball_seen = np.isfinite(ball_xy_contact).all(axis=1)

def smooth_ball_rts(points, fps, measurement_sigma, process_sigma):
    dt = 1 / fps
    transition = np.eye(4)
    transition[0, 2] = dt
    transition[1, 3] = dt
    observation = np.eye(2, 4)
    acceleration_map = np.r_[np.eye(2) * dt ** 2 / 2, np.eye(2) * dt]
    process_noise = acceleration_map @ acceleration_map.T * process_sigma ** 2
    measurement_noise = np.eye(2) * measurement_sigma ** 2
    state = np.r_[points[0], (points[1] - points[0]) / dt]
    covariance = np.diag([measurement_sigma ** 2, measurement_sigma ** 2, 100.0, 100.0])
    predicted = np.empty((len(points), 4))
    filtered = np.empty_like(predicted)
    predicted_cov = np.empty((len(points), 4, 4))
    filtered_cov = np.empty_like(predicted_cov)
    for i, point in enumerate(points):
        if i:
            state = transition @ state
            covariance = transition @ covariance @ transition.T + process_noise
        predicted[i] = state
        predicted_cov[i] = covariance
        gain = np.linalg.solve(observation @ covariance @ observation.T + measurement_noise, (covariance @ observation.T).T).T
        state = state + gain @ (point - observation @ state)
        residual_map = np.eye(4) - gain @ observation
        covariance = residual_map @ covariance @ residual_map.T + gain @ measurement_noise @ gain.T
        filtered[i] = state
        filtered_cov[i] = covariance
    smoothed = filtered.copy()
    for i in range(len(points) - 2, -1, -1):
        gain = np.linalg.solve(predicted_cov[i + 1].T, (filtered_cov[i] @ transition.T).T).T
        smoothed[i] += gain @ (smoothed[i + 1] - predicted[i + 1])
    return smoothed
starts = np.flatnonzero(ball_seen & ~np.r_[False, ball_seen[:-1]])
ends = np.flatnonzero(ball_seen & ~np.r_[ball_seen[1:], False])
acceleration = np.full(N_FRAMES, np.nan)
nearest_distance = np.full(N_FRAMES, np.nan)
nearest_offset = np.full(N_FRAMES, np.nan)
eligible = np.zeros(N_FRAMES, dtype=bool)
edge_guard = round(EDGE_GUARD_SECONDS * FPS)
for first, last in zip(starts, ends):
    idx = np.arange(first, last + 1)
    if len(idx) / FPS < MIN_RUN_SECONDS:
        continue
    state = smooth_ball_rts(ball_xy_contact[idx], FPS, MEASUREMENT_SIGMA_M, PROCESS_SIGMA_M_S2)
    acceleration[idx] = np.linalg.norm(np.gradient(state[:, 2:], 1 / FPS, axis=0), axis=1)
    distances = np.linalg.norm(player_xy_contact[idx] - state[:, None, :2], axis=2)
    distances = np.where(np.isfinite(distances), distances, np.inf)
    distance_min = distances.min(axis=1)
    offset_min = distances.argmin(axis=1).astype(float)
    offset_min[~np.isfinite(distance_min)] = np.nan
    distance_min[~np.isfinite(distance_min)] = np.nan
    nearest_distance[idx] = distance_min
    nearest_offset[idx] = offset_min
    if len(idx) > 2 * edge_guard:
        eligible[idx[edge_guard:-edge_guard]] = True
acc_series = pd.Series(acceleration)
local_peak = (acc_series.gt(acc_series.shift()) & acc_series.ge(acc_series.shift(-1))).to_numpy()
candidate_mask = eligible & local_peak & (acceleration >= PEAK_ACCELERATION_M_S2) & (nearest_distance <= NEAR_DISTANCE_M)
contact_candidate_side = np.full(N_FRAMES, '', dtype='<U1')
contact_candidate_side[candidate_mask] = track_sides[nearest_offset[candidate_mask].astype(int)]
HALVES = [('1. Halbzeit', 11185, 90651), ('2. Halbzeit', 110808, 190197)]

def classify_from_contacts(baseline, event_side, halves):
    prediction = baseline.copy()
    uses_contact_state = np.zeros(len(baseline), dtype=bool)
    for _, first, last in halves:
        last_team = None
        for frame in range(first, last + 1):
            if baseline[frame] == 'D':
                last_team = None
                continue
            if event_side[frame] in ('H', 'A'):
                last_team = event_side[frame]
            if last_team is not None:
                prediction[frame] = last_team
                uses_contact_state[frame] = True
    return (prediction, uses_contact_state)
baseline_original = pd.read_csv(CONTACT_DIR / 'submissions/baseline_smooth50.csv')['label'].to_numpy()
replayed_contact, uses_contact = classify_from_contacts(baseline_original, contact_candidate_side, HALVES)
original_contact = contact_04['label'].to_numpy()
n_differences = np.count_nonzero(replayed_contact != original_contact)
print('Kontaktkandidaten:', candidate_mask.sum())
print('Davon Barcelona:', (contact_candidate_side == 'H').sum())
print('Davon Athletic:', (contact_candidate_side == 'A').sum())
print('Abweichungen zu Notebook 04:', n_differences)


contact_frames = np.flatnonzero(candidate_mask)
contact_offsets = nearest_offset[contact_frames].astype(int)
contacts_table = pd.DataFrame({"frame": contact_frames, "team": contact_candidate_side[contact_frames],
                               "track_slot_id": np.asarray(track_ids)[contact_offsets],
                               "rts_ball_player_distance_m": nearest_distance[contact_frames],
                               "ball_acceleration_m_s2": acceleration[contact_frames],
                               "active_in_original_contact_state": uses_contact[contact_frames]})
contacts_table.insert(0, "reference_id", [f"C{i:05d}" for i in range(len(contacts_table))])
contacts_table.to_csv(TABLES / "original_tracking_contacts.csv", index=False)

## 5. Bestehende Besitzgrenzen und räumliche Kandidaten

Physics v3 und Spatial Hybrid v1 liefern Modellgrenzen. Räumliche Crossings verwenden denselben Median 11/min8 wie v1 und werden auf LIVE innerhalb der Halbzeiten begrenzt. Die vier Referenzlisten entstehen unabhängig von Videoklasse und Score.

In [ ]:
physics = pd.read_csv(PROJECT_DIR / "artifacts/spatial_hybrid_v1/references/physics_v3_reference.csv")
hybrid = pd.read_csv(PROJECT_DIR / "artifacts/spatial_hybrid_v1/references/spatial_hybrid_v1.csv")
with np.load(PROJECT_DIR / "artifacts/spatial_hybrid_v1/inputs/spatial_hybrid_inputs_v1.npz", allow_pickle=False) as blob:
    status = blob["status"].copy()
    proximity = blob["proximity_margin_H"].copy()
active = np.zeros(N_FRAMES, bool)
for _, first, last in HALVES:
    active[first:last + 1] = True
live = active & (status == "L")


def model_switches(labels, prefix):
    previous, current = labels[:-1], labels[1:]
    frames = np.flatnonzero(live[:-1] & live[1:] & np.isin(previous, ["H", "A"]) & np.isin(current, ["H", "A"]) & (previous != current)) + 1
    return pd.DataFrame({"reference_id": [f"{prefix}{i:05d}" for i in range(len(frames))], "frame": frames,
                         "from_team": labels[frames - 1], "team": labels[frames]})


physics_switches = model_switches(physics.label.to_numpy(), "P")
hybrid_switches = model_switches(hybrid.label.to_numpy(), "H")
margin_smooth = pd.Series(proximity).rolling(11, center=True, min_periods=8).median().to_numpy()
sign = np.zeros(N_FRAMES, np.int8)
sign[margin_smooth > 0], sign[margin_smooth < 0] = 1, -1
spatial_frames = np.flatnonzero(sign[1:] * sign[:-1] == -1) + 1
spatial_all = pd.DataFrame({"reference_id": [f"S{i:05d}" for i in range(len(spatial_frames))], "frame": spatial_frames,
                            "team": np.where(sign[spatial_frames] > 0, "H", "A"), "predicted_live": live[spatial_frames]})
references = {"tracking_contact": contacts_table, "physics_v3_switch": physics_switches,
              "spatial_hybrid_v1_switch": hybrid_switches,
              "spatial_live_crossing": spatial_all.loc[spatial_all.predicted_live].copy()}
for name, table in references.items():
    table.to_csv(TABLES / f"{name}_reference_events.csv", index=False)
spatial_all.to_csv(TABLES / "spatial_crossings_all_statuses.csv", index=False)
video["predicted_live"] = live[video.frame.to_numpy()]
print({name: len(table) for name, table in references.items()})
source_names = list(references)
class_names = sorted(video.video_event_type.unique())
clip_indices = {scene.scene_id: np.flatnonzero(video.scene_id.eq(scene.scene_id)) for scene in clips.itertuples()}
reference_local = {(scene.scene_id, name): table.loc[table.frame.between(scene.first_frame - 50, scene.stop_frame_exclusive - 1 + 50)]
                   .sort_values(["frame", "reference_id"], kind="stable").reset_index(drop=True)
                   for scene in clips.itertuples() for name, table in references.items()}
class_masks = {"ALL": np.ones(len(video), bool), **{name: video.video_event_type.eq(name).to_numpy() for name in class_names}}
print("Referenzen werden im größten Tracking-Zeitfenster auch außerhalb der Clipschnittkante berücksichtigt.")

## 6. Zeitliche Eins-zu-eins-Zuordnung

Die lineare Zuordnung priorisiert maximale Paarzahl vor minimalem Zeitabstand. Mehrere zulässige Kontakte oder Konkurrenz anderer Events werden als mehrdeutig markiert. Positives Delta bedeutet: Trackingkontakt später als Videovorhersage.

In [ ]:
def assign_events(video_frames, reference_frames, tolerance_frames):
    delta = reference_frames[None, :] - video_frames[:, None]
    allowed = np.abs(delta) <= tolerance_frames
    n_video, n_reference = allowed.shape
    matched = np.full(n_video, -1, int)
    if not n_video or not n_reference:
        return matched, np.zeros(n_video, int), np.zeros(n_video, bool)
    unmatched_cost = (n_video + 1) * (tolerance_frames + 1)
    cost = np.where(allowed, np.abs(delta), unmatched_cost * (n_video + 2)).astype(float)
    cost = np.column_stack([cost, np.full((n_video, n_video), unmatched_cost)])
    rows, columns = linear_sum_assignment(cost)
    selected = columns < n_reference
    matched[rows[selected]] = columns[selected]
    degrees = allowed.sum(axis=1)
    competition = (allowed & (allowed.sum(axis=0) > 1)[None, :]).any(axis=1)
    return matched, degrees, (degrees > 1) | competition


def match_scenario(event_frames, collect_pairs=False):
    metrics, pair_records = [], []
    event_live = live[event_frames]
    for name in source_names:
        for window in WINDOWS_S:
            matched_frames = np.full(len(video), np.nan)
            degree = np.zeros(len(video), int)
            ambiguous = np.zeros(len(video), bool)
            for scene in clips.itertuples():
                indices = clip_indices[scene.scene_id]
                reference_table = reference_local[scene.scene_id, name]
                assignment, local_degree, local_ambiguous = assign_events(event_frames[indices], reference_table.frame.to_numpy(), round(window * FPS))
                degree[indices], ambiguous[indices] = local_degree, local_ambiguous
                chosen = assignment >= 0
                matched_frames[indices[chosen]] = reference_table.frame.to_numpy()[assignment[chosen]]
                if collect_pairs:
                    for offset, event_index in enumerate(indices):
                        reference_index = assignment[offset]
                        reference_row = reference_table.iloc[reference_index] if reference_index >= 0 else None
                        pair_records.append({"event_id": video.iloc[event_index].event_id, "scene_id": scene.scene_id,
                            "video_event_type": video.iloc[event_index].video_event_type, "video_frame": int(event_frames[event_index]),
                            "source": name, "window_s": window, "candidate_count": int(local_degree[offset]),
                            "ambiguous": bool(local_ambiguous[offset]), "matched": bool(reference_index >= 0),
                            "reference_id": reference_row.reference_id if reference_row is not None else None,
                            "reference_frame": int(reference_row.frame) if reference_row is not None else None,
                            "reference_team": reference_row.team if reference_row is not None else None,
                            "time_delta_s": (reference_row.frame - event_frames[event_index]) / FPS if reference_row is not None else np.nan})
            for event_class, mask in class_masks.items():
                matched_mask = np.isfinite(matched_frames[mask])
                deltas = (matched_frames[mask][matched_mask] - event_frames[mask][matched_mask]) / FPS
                metrics.append({"source": name, "window_s": window, "event_class": event_class,
                    "n_evaluable": int(mask.sum()), "n_matched": int(matched_mask.sum()),
                    "match_fraction": float(matched_mask.mean()), "unmatched_fraction": float(1 - matched_mask.mean()),
                    "median_abs_delta_s": float(np.median(np.abs(deltas))) if len(deltas) else np.nan,
                    "median_signed_delta_s": float(np.median(deltas)) if len(deltas) else np.nan,
                    "n_ambiguous": int(ambiguous[mask].sum()), "predicted_live_fraction": float(event_live[mask].mean())})
    return metrics, pair_records


observed_records, pair_records = match_scenario(video.frame.to_numpy(), collect_pairs=True)
observed = pd.DataFrame(observed_records)
pairs = pd.DataFrame(pair_records)
observed.to_csv(TABLES / "observed_matching_metrics.csv", index=False)
pairs.to_csv(TABLES / "one_to_one_event_matches.csv", index=False)
display(observed.loc[observed.event_class.ne("ALL") & observed.source.eq("tracking_contact")].round(3))
print("Kein Match bezeichnet fehlende Übereinstimmung mit dieser Referenzliste, keinen bewiesenen Videofehler.")

## 7. Negativkontrolle

Alle Events eines Clips werden gemeinsam zirkulär verschoben. Die geometrisch abgedeckte Strecke umfasst 11,6 s, der Mindestabstand beträgt 2,4 s. Klassenanzahlen bleiben erhalten. 64 Vergleiche werden gemeinsam nach Holm korrigiert. Die Tail-Anteile beschreiben dieses begrenzte Nullmodell; sie sind keine Präzisionsmessung oder exakt kalibrierten konfirmatorischen p-Werte.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
shift_frames = np.empty((N_SHIFTS, len(clips)), int)
shift_domains = {}
for column, scene in enumerate(clips.itertuples()):
    covered = np.flatnonzero(score_data[scene.scene_id]["window_coverage"] > 0)
    ring_frames = len(covered) * STRIDE
    minimum = int(np.ceil(MIN_SHIFT_S * FPS / STRIDE))
    allowed_shifts = np.arange(minimum, len(covered) - minimum + 1) * STRIDE
    shift_frames[:, column] = rng.choice(allowed_shifts, N_SHIFTS)
    shift_domains[scene.scene_id] = {"ring_frames": ring_frames, "ring_seconds": ring_frames / FPS,
                                   "possible_shifts": len(allowed_shifts), "minimum_shift_frames": int(allowed_shifts.min())}
np.savez_compressed(OUT / "manifests/negative_control_shifts.npz", shifts_frames=shift_frames, scene_ids=clips.scene_id.to_numpy(dtype=str))
null_rows = []
original_frames = video.frame.to_numpy()
for repetition in range(N_SHIFTS):
    shifted = original_frames.copy()
    for column, scene in enumerate(clips.itertuples()):
        indices = clip_indices[scene.scene_id]
        ring = shift_domains[scene.scene_id]["ring_frames"]
        shifted[indices] = scene.first_frame + (original_frames[indices] - scene.first_frame + shift_frames[repetition, column]) % ring
    metrics, _ = match_scenario(shifted)
    null_rows.extend({"repetition": repetition, **row} for row in metrics)
    if (repetition + 1) % 250 == 0:
        print("Negativkontrolle:", repetition + 1, "/", N_SHIFTS, flush=True)
null = pd.DataFrame(null_rows)
null.to_csv(TABLES / "negative_control_metrics.csv", index=False)
comparison_rows = []
for row in observed.to_dict("records"):
    distribution = null.loc[null.source.eq(row["source"]) & null.window_s.eq(row["window_s"]) & null.event_class.eq(row["event_class"])]
    comparison_rows.append({**row, "null_match_median": float(distribution.match_fraction.median()),
        "null_match_q025": float(distribution.match_fraction.quantile(.025)),
        "null_match_q975": float(distribution.match_fraction.quantile(.975)),
        "excess_match_fraction": row["match_fraction"] - float(distribution.match_fraction.mean()),
        "empirical_upper_tail": (1 + int(distribution.n_matched.ge(row["n_matched"]).sum())) / (N_SHIFTS + 1),
        "null_median_abs_delta_s": float(distribution.median_abs_delta_s.median()) if distribution.median_abs_delta_s.notna().any() else np.nan})
comparison = pd.DataFrame(comparison_rows)
p_values = comparison.empirical_upper_tail.to_numpy()
order = np.argsort(p_values, kind="stable")
adjusted = np.minimum(1, np.maximum.accumulate(p_values[order] * (len(p_values) - np.arange(len(p_values)))))
comparison["holm_adjusted_tail"] = np.empty(len(comparison))
comparison.loc[order, "holm_adjusted_tail"] = adjusted
comparison.to_csv(TABLES / "observed_vs_shifted_comparison.csv", index=False)
(OUT / "manifests/negative_control_domains.json").write_text(json.dumps(shift_domains, indent=2))
display(comparison.loc[comparison.source.eq("tracking_contact"), ["event_class", "window_s", "n_matched", "match_fraction", "null_match_median", "excess_match_fraction", "empirical_upper_tail", "holm_adjusted_tail"]].round(3))

## 8. Teamindizien und Featureexport

Trackingkonsens verlangt LIVE, genau ein Team innerhalb der ursprünglichen 2-m-Grenze, passende Kontaktzuordnung und Übereinstimmung beider Besitzmodelle. Das sind abhängige Trackingindizien, keine unabhängigen Video-Teamlabels. Native Teams bleiben unbekannt, Spieler-IDs leer. Eine Zeile je Event wird für Notebook 05 gespeichert.

In [ ]:
primary = pairs.loc[pairs.source.eq("tracking_contact") & pairs.window_s.eq(FEATURE_WINDOW_S)].set_index("event_id")
wide = pairs.loc[pairs.source.eq("tracking_contact") & pairs.window_s.eq(max(WINDOWS_S))].set_index("event_id")
physics_labels, hybrid_labels = physics.label.to_numpy(), hybrid.label.to_numpy()
feature_rows = []
contact_times = contacts_table.frame.to_numpy()
for event in video.itertuples():
    frame = int(event.frame)
    original_match = primary.loc[event.event_id]
    near_contacts = contacts_table.loc[np.abs(contact_times - frame) <= round(FEATURE_WINDOW_S * FPS)]
    near_contacts_wide = contacts_table.loc[np.abs(contact_times - frame) <= round(max(WINDOWS_S) * FPS)]
    if len(near_contacts_wide):
        contact_distances = np.abs(near_contacts_wide.frame.to_numpy() - frame)
        closest_contact = near_contacts_wide.iloc[int(np.argmin(contact_distances))]
        nearest_frame, nearest_delta, nearest_team = int(closest_contact.frame), (closest_contact.frame - frame) / FPS, closest_contact.team
        nearest_tied = int((contact_distances == contact_distances.min()).sum()) > 1
    else:
        nearest_frame, nearest_delta, nearest_team, nearest_tied = None, np.nan, None, False
    raw_ball_available = bool(ball_seen[frame])
    distances = np.linalg.norm(player_xy_contact[frame] - ball_xy_contact[frame], axis=1) if raw_ball_available else np.full(len(track_ids), np.nan)
    finite = np.isfinite(distances)
    nearby = finite & (distances <= NEAR_DISTANCE_M)
    nearby_teams = set(track_sides[nearby])
    closest_slot = int(np.nanargmin(distances)) if finite.any() else None
    geometry_team = next(iter(nearby_teams)) if len(nearby_teams) == 1 else None
    contact_teams = set(near_contacts.team)
    contact_team = next(iter(contact_teams)) if len(contact_teams) == 1 else None
    physics_team = physics_labels[frame] if physics_labels[frame] in ("H", "A") else None
    hybrid_team = hybrid_labels[frame] if hybrid_labels[frame] in ("H", "A") else None
    consensus = (live[frame] and geometry_team is not None and geometry_team == contact_team == physics_team == hybrid_team
                 and original_match.matched and not original_match.ambiguous)
    if consensus:
        attribution, inferred_team, confidence_label = "tracking_consensus", geometry_team, "consistent_but_unvalidated"
    elif nearby_teams or contact_teams:
        attribution, inferred_team, confidence_label = "ambiguous", None, "ambiguous"
    else:
        attribution, inferred_team, confidence_label = "unavailable", None, "no_actor_evidence"
    reasons = []
    if not raw_ball_available: reasons.append("ball_missing_at_event")
    if not nearby_teams: reasons.append("no_visible_slot_within_original_2m")
    if len(nearby_teams) > 1: reasons.append("opposing_teams_near_ball")
    if not contact_teams: reasons.append("no_contact_candidate_within_0_5s")
    if len(contact_teams) > 1: reasons.append("opposing_contact_teams_within_0_5s")
    if original_match.ambiguous: reasons.append("temporal_assignment_ambiguous")
    if not original_match.matched: reasons.append("no_one_to_one_contact")
    if not live[frame]: reasons.append("predicted_status_not_live")
    if physics_team != hybrid_team: reasons.append("physics_hybrid_team_disagreement")
    if geometry_team and contact_team and geometry_team != contact_team: reasons.append("geometry_contact_team_disagreement")
    if geometry_team and any(t is not None and t != geometry_team for t in [physics_team, hybrid_team]): reasons.append("geometry_possession_team_disagreement")
    context = slice(max(0, frame - 2 * FPS), min(N_FRAMES, frame + 2 * FPS + 1))
    feature_rows.append({"event_id": event.event_id, "scene_id": event.scene_id, "frame": frame,
        "video_event_type": event.video_event_type, "video_event_score": event.video_event_score,
        "video_event_timestamp": event.video_timestamp_s, "source_clip": event.source_clip,
        "nearest_contact_frame": nearest_frame, "contact_time_delta_s": nearest_delta, "nearest_contact_team": nearest_team,
        "nearest_contact_tied": nearest_tied, "nearest_contact_max_radius_s": max(WINDOWS_S),
        "one_to_one_contact_frame": original_match.reference_frame, "one_to_one_contact_delta_s": original_match.time_delta_s,
        "contact_candidates_0_5s": int(original_match.candidate_count), "contact_candidates_2s": int(wide.loc[event.event_id].candidate_count),
        "contact_matching_ambiguous_0_5s": bool(original_match.ambiguous), "video_event_team": "unknown", "player_track_id": None,
        "tracking_attributed_team": inferred_team, "team_attribution_status": attribution,
        "team_attribution_source": "raw_geometry+original_contacts+frozen_models; not video" if consensus else "tracking_evidence_insufficient_or_conflicting",
        "team_attribution_confidence": confidence_label, "team_attribution_reasons": ";".join(reasons) if reasons else "strict_tracking_consensus", "raw_ball_available": raw_ball_available,
        "nearest_track_slot_id": track_ids[closest_slot] if closest_slot is not None else None,
        "nearest_track_team": track_sides[closest_slot] if closest_slot is not None else None,
        "nearest_raw_player_distance_m": float(distances[closest_slot]) if closest_slot is not None else np.nan,
        "nearby_track_slots_2m": int(nearby.sum()), "nearby_teams_2m": ";".join(sorted(nearby_teams)),
        "raw_geometric_team_candidate": geometry_team, "nearby_contact_team_candidate": contact_team,
        "tracking_status": status[frame], "physics_v3_team": physics_team, "spatial_hybrid_v1_team": hybrid_team,
        "physics_v3_label": physics_labels[frame], "spatial_hybrid_v1_label": hybrid_labels[frame],
        "contact_detector_eligible_at_frame": bool(eligible[frame]),
        "contact_detector_eligible_any_0_5s": bool(eligible[max(0, frame - 12):min(N_FRAMES, frame + 13)].any()),
        "physics_stable_team_2s": bool(physics_team and (physics_labels[context] == physics_team).all() and live[context].all()),
        "hybrid_stable_team_2s": bool(hybrid_team and (hybrid_labels[context] == hybrid_team).all() and live[context].all())})
features = pd.DataFrame(feature_rows)
for source in ["physics_v3_switch", "spatial_hybrid_v1_switch", "spatial_live_crossing"]:
    local = pairs.loc[pairs.source.eq(source) & pairs.window_s.eq(FEATURE_WINDOW_S), ["event_id", "reference_frame", "time_delta_s", "ambiguous"]]
    local = local.rename(columns={col: f"{source}_{col}" for col in local if col != "event_id"})
    features = features.merge(local, on="event_id", validate="one_to_one")
features.to_csv(OUT / "tdeed_tracking_event_features.csv", index=False)
features.loc[features.contact_candidates_0_5s.eq(0)].to_csv(TABLES / "events_without_any_contact_0_5s.csv", index=False)
novelty = features.groupby("video_event_type").agg(n_video=("event_id", "size"),
    no_contact_candidate_0_5s=("contact_candidates_0_5s", lambda x: int(x.eq(0).sum())),
    unmatched_one_to_one_0_5s=("one_to_one_contact_frame", lambda x: int(x.isna().sum())),
    no_contact_candidate_2s=("contact_candidates_2s", lambda x: int(x.eq(0).sum())),
    raw_ball_missing=("raw_ball_available", lambda x: int((~x).sum())),
    stable_hybrid_context_2s=("hybrid_stable_team_2s", "sum"))
novelty.to_csv(TABLES / "additional_temporal_proposals_by_class.csv")
attribution_counts = features.team_attribution_status.value_counts().rename_axis("status").rename("events")
attribution_counts.to_csv(TABLES / "team_attribution_summary.csv")
features.assign(reason=features.team_attribution_reasons.str.split(";")).explode("reason").groupby("reason").size().rename("events").to_csv(TABLES / "team_attribution_reasons.csv")
coverage_rows = []
for scene in clips.itertuples():
    local = features.loc[features.scene_id.eq(scene.scene_id)]
    arrays = score_data[scene.scene_id]
    coverage_rows.append({"scene_id": scene.scene_id, "events": len(local), "video_frames": scene.clip_frames,
        "geometric_model_frame_coverage_fraction": float((arrays["window_coverage"] > 0).sum() * STRIDE / scene.clip_frames),
        "score_frame_support_fraction": float((arrays["support"] > 0).sum() * STRIDE / scene.clip_frames),
        "ball_available_frame_fraction": float(ball_seen[scene.first_frame:scene.stop_frame_exclusive].mean()),
        "original_detector_eligible_frame_fraction": float(eligible[scene.first_frame:scene.stop_frame_exclusive].mean()),
        "event_ball_available_fraction": float(local.raw_ball_available.mean()),
        "event_any_detector_coverage_0_5s_fraction": float(local.contact_detector_eligible_any_0_5s.mean()),
        "event_predicted_live_fraction": float(local.tracking_status.eq("L").mean()),
        "unmatched_events_0_5s": int(local.one_to_one_contact_frame.isna().sum()),
        "tracking_consensus_events": int(local.team_attribution_status.eq("tracking_consensus").sum())})
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(TABLES / "coverage_by_clip.csv", index=False)
display(novelty)
display(attribution_counts.to_frame())
display(coverage.round(3))
print("Semantisch sicherer Videoakteur: nicht unabhängig bestätigt; native Teams bleiben unknown, Spieler-ID leer.")

## 9. Zeitachsen und Vergleich mit den Verschiebungen

Gemeinsame Markierungen zeigen zeitliche Nähe. Die grauen D-Abschnitte stammen aus der Modellvorhersage.

In [ ]:
team_colors = {"H": "tab:blue", "A": "tab:red"}
fig, axes = plt.subplots(5, 1, figsize=(14, 13), constrained_layout=True)
fig.suptitle("Videoereignisse ohne Teamzuordnung; Trackingreferenzen: H blau, A rot", fontsize=12)
for ax, scene in zip(axes, clips.itertuples()):
    local = video.loc[video.scene_id.eq(scene.scene_id)]
    for level, name in enumerate(class_names):
        part = local.loc[local.video_event_type.eq(name)]
        ax.scatter((part.frame - scene.first_frame) / FPS, np.full(len(part), level), s=35 + 100 * part.video_event_score, label=name)
    for level, (source, label) in enumerate(zip(source_names, ["Trackingkontakt", "Physics-H/A", "Hybrid-H/A", "Räumlich LIVE"]), start=3):
        part = references[source].loc[references[source].frame.between(scene.first_frame, scene.stop_frame_exclusive - 1)]
        for side, points in part.groupby("team"):
            ax.scatter((points.frame - scene.first_frame) / FPS, np.full(len(points), level), marker="|", s=130, color=team_colors[side])
    dead = status[scene.first_frame:scene.stop_frame_exclusive] == "D"
    for indices in np.split(np.flatnonzero(dead), np.flatnonzero(np.diff(np.flatnonzero(dead)) > 1) + 1):
        if len(indices):ax.axvspan(indices[0] / FPS, (indices[-1] + 1) / FPS, color="gray", alpha=.10)
    ax.set(title=f"{scene.scene_id}: T-DEED und bestehende Trackingreferenzen", xlim=(-.08, 12.08), ylim=(-.6, 6.6),
           yticks=range(7), yticklabels=[*class_names, "Trackingkontakt", "Physics-H/A", "Hybrid-H/A", "Räumlich LIVE"],
           xlabel="Lokale Clipzeit [s]; graue Abschnitte = vorhergesagtes D")
fig.savefig(FIG / "02_joint_event_timelines.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, name in zip(axes, class_names):
    values = features.loc[features.video_event_type.eq(name), "one_to_one_contact_delta_s"].dropna()
    ax.hist(values, bins=np.linspace(-.52, .52, 14), color="tab:blue")
    ax.axvline(0, color="black", lw=1)
    ax.set(title=f"{name}: {len(values)} Paare bei ±0,5s", xlabel="Tracking minus Video [s]", ylabel="Paaranzahl")
fig.savefig(FIG / "03_contact_time_differences.png", dpi=150)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for ax, source in zip(axes.flat, source_names):
    part = comparison.loc[comparison.source.eq(source) & comparison.event_class.eq("ALL")].sort_values("window_s")
    x = np.arange(len(part))
    ax.fill_between(x, part.null_match_q025.to_numpy(), part.null_match_q975.to_numpy(), color="gray", alpha=.25, label="Verschoben: 2,5–97,5%")
    ax.plot(x, part.null_match_median, "o--", color="gray", label="Verschoben: Median")
    ax.plot(x, part.match_fraction, "o-", color="tab:blue", label="Tatsächliche Zeitpunkte")
    ax.set(title=source, xticks=x, xticklabels=[str(v) for v in part.window_s], xlabel="Halbfenster [s]", ylabel="Anteil eins zu eins zugeordnet", ylim=(-.03, 1.03))
axes.flat[0].legend(fontsize=8)
fig.savefig(FIG / "04_observed_vs_shifted.png", dpi=150)
plt.show()
print("Beobachtung: Die Richtung und Stärke gegenüber der Nullverteilung sind tabellarisch zu beurteilen; semantische Eventqualität wurde nicht gemessen.")

In [ ]:
dictionary = """# Herkunft der Featuretabelle

Eine Zeile pro finalem T-DEED-Pilotereignis; keine dichte Vollspielprognose und keine neue Zielvariable.

| Feldgruppe | Bedeutung |
|---|---|
| event_id / scene_id / source_clip | Stabile Diagnose-ID und vorhandener Inferenzclip. |
| frame / video_event_timestamp | Geprüfte globale25-FPS-Adresse / Originalvideo-PTS in Sekunden; Eventraster80ms. |
| video_event_type / video_event_score | Native Klasse und unveränderter unkalibrierter Post-Soft-NMS-Score. |
| nearest_contact_frame / contact_time_delta_s | Zeitlich nächster Originalkontakt innerhalb±2s; Kontakt minus Video. Bei Gleichstand früherer sortierter Kontakt und nearest_contact_tied=True. |
| one_to_one_contact_frame / delta | Optimale gemeinsame Zuordnung bei±0,5s; kann wegen Konkurrenz vom nächsten Kontakt abweichen oder fehlen. |
| contact_candidates_* / contact_matching_ambiguous_* | Anzahl zulässiger Kandidaten und Mehrdeutigkeit des Abstandsgraphen. |
| video_event_team / player_track_id | Native Teamunterstützung fehlt: unknown / leer. Nicht mit Trackingzuordnung überschreiben. |
| tracking_attributed_team / status / source / confidence | Unvalidiertes H/A nur bei strenger Trackingkonsistenz; sonst leer. Status tracking_consensus, ambiguous, unavailable. Confidence ist kategoriale Evidenzqualität, keine numerische Wahrscheinlichkeit. team_attribution_reasons nennt jede fehlende oder widersprüchliche Voraussetzung. |
| nearest_track_slot_id / nearest_track_team / nearest_raw_player_distance_m | Nächster sichtbarer roher Trackslot und seine Metadaten; kein bestätigter Eventakteur. Kann weit vom Ball sein, siehe Abstand. |
| nearby_track_slots_2m / nearby_teams_2m | Alle sichtbaren Slots/Teams innerhalb der ursprünglichen2-m-Nähegrenze. Mehrere Slots sind nicht zwingend verschiedene identifizierte Personen. |
| raw_geometric_team_candidate / nearby_contact_team_candidate | Getrennte Teamindizien, jeweils nur bei genau einem vorhandenen Team. |
| tracking_status / physics_v3_* / spatial_hybrid_v1_* | Bestehende Modellvorhersagen; Teamfelder bei D/X leer, Labelfelder behalten D/X. Keine GT. |
| raw_ball_available / contact_detector_eligible_* | Daten-/Originaldetektorabdeckung; Ausfall ist kein Beleg für falsches Videoevent. |
| *_stable_team_2s | Bestehende Besitzsequenz und Status im ±2-s-Kontext konstant H/A und LIVE; kein Wahrheitslabel. |
| *_switch_reference_frame / *_time_delta_s / *_ambiguous | Separate Eins-zu-eins-Zuordnungen bei±0,5s zu den anderen Referenzquellen. |

Fehlende numerische Informationen sind CSV-Leerwerte/NaN. Kein Score oder Label wurde mit öffentlicher GT korrigiert. Die Konsensregel und alle Fenster stehen vorab im Notebook; die Tabelle ist nur für weitere freizugebende Diagnose/Integration gedacht.
"""

(OUT / "FEATURE_DICTIONARY.md").write_text(dictionary)
primary_summary = comparison.loc[comparison.source.eq("tracking_contact") & comparison.window_s.eq(FEATURE_WINDOW_S)]
display(primary_summary[["event_class", "n_matched", "match_fraction", "null_match_median", "median_abs_delta_s", "holm_adjusted_tail"]].round(3))
print("Trackingkonsens:", int(features.team_attribution_status.eq("tracking_consensus").sum()), "von", len(features))

## Ergebnis

Im bisherigen Lauf wurden 19 von 47 Events innerhalb ±0,5 s einem Kontakt zugeordnet; bei verschobenen Zeitpunkten lag der mediane Anteil bei 0,362. Keiner der 64 Vergleiche lag nach Holm unter 0,05. Sieben Events erfüllen die unvalidierte Trackingkonsensregel. Ein zusätzlicher Nutzen für die Ballbesitzklassifikation ist damit nicht belegt. Die technische Fusion folgt in Notebook 05.